# Subtask 1 — Pixel Tabular Baseline (Notebook 2 of 5)

**Goal:** Train the fastest valid model to establish an Accuracy±1 baseline on the val set.

**Approach:** Sample pixels from training patches, extract temporal features, train
`HistGradientBoostingClassifier` and `ExtraTreesClassifier`, run inference patch-by-patch on val.

**HITL gate:** After running, VB reports val Accuracy±1:
- ≥ 0.90 → proceed to Notebook 5 (submission) with this model
- < 0.90 → proceed to Notebook 3 (U-Net)

**Key parameters to tune via HITL:**
- `PIXELS_PER_CLASS` — how many pixels to sample per class
- `TEMPORAL_OPTION` — which feature set to use
- `USE_ORDINAL_REGRESSOR` — True = round regression output; False = direct classifier

## 0. Setup

In [ ]:
# !pip install -q rasterio scikit-learn joblib tqdm pillow

import csv
import json
import pickle
import zipfile
from collections import Counter
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin
from urllib.request import urlopen

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ── CONFIG ────────────────────────────────────────────────────────────────────
DATA_DIR        = None          # local path or None for HF streaming
LABEL_NAME      = "viticulture"
HF_BASE         = "https://huggingface.co/datasets/m-sakka/agripotential/resolve/main/"
OUT_DIR         = Path("../results/subtask1/baseline")
CKPT_DIR        = Path("../results/subtask1/checkpoints")
VIS_DIR         = Path("../results/subtask1/val_vis")
for d in [OUT_DIR, CKPT_DIR, VIS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Feature / sampling config
PIXELS_PER_CLASS  = 20_000   # total samples per class (adjust to fit RAM)
RANDOM_SEED       = 42
USE_ORDINAL_REGRESSOR = False  # set True to compare rounded regression

# Ordinal label soft-encoding weights for adjacent classes (used in evaluation display)
CLASS_NAMES  = ["Very Low", "Low", "Medium", "High", "Very High"]
CLASS_COLORS = ["#d73027", "#fc8d59", "#fee08b", "#91cf60", "#1a9850"]

rng = np.random.default_rng(RANDOM_SEED)
print("Setup complete.")

## 1. Load Metadata

In [ ]:
import rasterio
from rasterio.windows import Window
import pandas as pd

def ref(name):
    if DATA_DIR:
        return str(Path(DATA_DIR) / name)
    return urljoin(HF_BASE, name)

def read_csv(name):
    r = ref(name)
    if r.startswith("http"):
        with urlopen(r, timeout=60) as h:
            lines = [l.decode("utf-8") for l in h]
    else:
        lines = Path(r).read_text().splitlines()
    return list(csv.DictReader(lines))

metadata   = read_csv("metadata.csv")
train_rows = read_csv("train.csv")
val_rows   = read_csv("val.csv")
test_rows  = read_csv("test.csv")

meta_df = pd.DataFrame(metadata)
meta_df["date"] = pd.to_datetime(
    meta_df[["year", "month", "day"]].astype(int)
)
meta_df = meta_df.sort_values("date").reset_index(drop=True)
sentinel_files = list(meta_df["filename"])
n_times = len(sentinel_files)
print(f"Timeframes: {n_times}, Train: {len(train_rows)}, Val: {len(val_rows)}, Test: {len(test_rows)}")

# Month index per timeframe (for seasonal grouping)
tf_months = list(meta_df["month"].astype(int))
SPRING_IDX = [i for i, m in enumerate(tf_months) if m in (3, 4, 5)]
SUMMER_IDX = [i for i, m in enumerate(tf_months) if m in (6, 7, 8)]
AUTUMN_IDX = [i for i, m in enumerate(tf_months) if m in (9, 10, 11)]
WINTER_IDX = [i for i, m in enumerate(tf_months) if m in (12, 1, 2)]
print(f"Spring frames: {len(SPRING_IDX)}, Summer: {len(SUMMER_IDX)}, "
      f"Autumn: {len(AUTUMN_IDX)}, Winter: {len(WINTER_IDX)}")

## 2. Feature Extraction Function

In [ ]:
N_BANDS = 10
BAND_NAMES = ["B1","B2","B3","B4","B5","B6","B7","B8","B8A","B9"]

def extract_patch_features(row_idx, col_idx, patch_size):
    """Return feature matrix of shape (H*W, n_features) for one patch."""
    win = Window(col_idx, row_idx, patch_size, patch_size)
    # Stack all timeframes: (T, B, H, W)
    frames = []
    for fname in sentinel_files:
        with rasterio.open(ref(fname)) as src:
            arr = src.read(window=win).astype(np.float32)  # (B, H, W)
        frames.append(arr)
    stack = np.stack(frames, axis=0)  # (T, B, H, W)
    T, B, H, W = stack.shape
    P = H * W  # pixels

    features = []
    feat_names = []

    # 1. Raw reflectance for all T×B (340 features if T=34, B=10)
    raw = stack.transpose(2, 3, 0, 1).reshape(P, T * B)  # (P, T*B)
    features.append(raw)
    feat_names += [f"t{t}_b{b}" for t in range(T) for b in range(B)]

    # 2. Per-band temporal statistics (mean, std, min, max) → 4*B
    pix = stack.reshape(T, B, P).transpose(2, 0, 1)  # (P, T, B)
    feat_tmean = pix.mean(axis=1)   # (P, B)
    feat_tstd  = pix.std(axis=1)
    feat_tmin  = pix.min(axis=1)
    feat_tmax  = pix.max(axis=1)
    features += [feat_tmean, feat_tstd, feat_tmin, feat_tmax]
    feat_names += ([f"tmean_b{b}" for b in range(B)] +
                   [f"tstd_b{b}"  for b in range(B)] +
                   [f"tmin_b{b}"  for b in range(B)] +
                   [f"tmax_b{b}"  for b in range(B)])

    # 3. Seasonal means per band
    for season_name, idx in [("spr", SPRING_IDX), ("sum", SUMMER_IDX),
                              ("aut", AUTUMN_IDX), ("win", WINTER_IDX)]:
        if idx:
            sea = pix[:, idx, :].mean(axis=1)  # (P, B)
        else:
            sea = np.zeros((P, B), dtype=np.float32)
        features.append(sea)
        feat_names += [f"{season_name}_b{b}" for b in range(B)]

    # 4. NDVI proxy (band 7=B8 NIR, band 3=B4 Red) across time: mean, std
    nir = pix[:, :, 7]  # (P, T)
    red = pix[:, :, 3]  # (P, T)
    denom = nir + red + 1e-6
    ndvi = (nir - red) / denom  # (P, T)
    features.append(ndvi.mean(axis=1, keepdims=True))
    features.append(ndvi.std(axis=1, keepdims=True))
    feat_names += ["ndvi_mean", "ndvi_std"]

    X = np.concatenate(features, axis=1)  # (P, n_feats)
    return X, feat_names

# Quick smoke-test on one patch
p0 = train_rows[0]
X0, feat_names = extract_patch_features(int(p0["row"]), int(p0["col"]), int(p0["patch_size"]))
print(f"Feature matrix shape: {X0.shape}  (pixels × features)")
print(f"Total features: {len(feat_names)}")
print(f"First 5 feature names: {feat_names[:5]}")

## 3. Stratified Pixel Sampling

In [ ]:
# Sample pixels from train patches, stratified by class.
# This avoids class imbalance overwhelming the model.

label_ref = ref(f"{LABEL_NAME}.tif")
X_parts, y_parts = [], []
counts_per_class = Counter()

with rasterio.open(label_ref) as lsrc:
    for patch in tqdm(train_rows, desc="Sampling train pixels"):
        if all(counts_per_class.get(c, 0) >= PIXELS_PER_CLASS for c in range(5)):
            break
        r, c, ps = int(patch["row"]), int(patch["col"]), int(patch["patch_size"])
        lbl = lsrc.read(1, window=Window(c, r, ps, ps)).astype(np.uint8).ravel()
        X_patch, _ = extract_patch_features(r, c, ps)
        for cls in range(5):
            need = PIXELS_PER_CLASS - counts_per_class.get(cls, 0)
            if need <= 0:
                continue
            mask = np.where(lbl == cls)[0]
            if len(mask) == 0:
                continue
            chosen = rng.choice(mask, size=min(need, len(mask)), replace=False)
            X_parts.append(X_patch[chosen])
            y_parts.append(lbl[chosen])
            counts_per_class[cls] += len(chosen)

X_train = np.concatenate(X_parts, axis=0)
y_train = np.concatenate(y_parts, axis=0)
print(f"Training set: {X_train.shape}, class counts: {dict(counts_per_class)}")

## 4. Train Models

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib

models = {}

print("Training HistGradientBoostingClassifier...")
hgbc = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, max_depth=6,
    l2_regularization=0.1, random_state=RANDOM_SEED, verbose=1
)
hgbc.fit(X_train, y_train)
models["hgbc"] = hgbc
joblib.dump(hgbc, CKPT_DIR / "hgbc_pixel.pkl")
print("Saved hgbc_pixel.pkl")

print("\nTraining ExtraTreesClassifier...")
etc = ExtraTreesClassifier(
    n_estimators=300, n_jobs=-1, class_weight="balanced",
    random_state=RANDOM_SEED, verbose=1
)
etc.fit(X_train, y_train)
models["etc"] = etc
joblib.dump(etc, CKPT_DIR / "etc_pixel.pkl")
print("Saved etc_pixel.pkl")

if USE_ORDINAL_REGRESSOR:
    print("\nTraining HistGradientBoostingRegressor (ordinal)...")
    hgbr = HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.05, max_depth=6,
        l2_regularization=0.1, random_state=RANDOM_SEED, verbose=1
    )
    hgbr.fit(X_train, y_train.astype(float))
    models["hgbr"] = hgbr
    joblib.dump(hgbr, CKPT_DIR / "hgbr_pixel.pkl")
    print("Saved hgbr_pixel.pkl")

## 5. Evaluate on Validation Set

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.colors as mcolors

def accuracy_pm1(y_true, y_pred):
    """Proportion of predictions within 1 ordinal class of truth."""
    return float(np.mean(np.abs(y_true.astype(int) - y_pred.astype(int)) <= 1))

def run_val(model, model_name, is_regressor=False):
    all_pred, all_true = [], []
    with rasterio.open(label_ref) as lsrc:
        for patch in tqdm(val_rows, desc=f"Val [{model_name}]"):
            r, c, ps = int(patch["row"]), int(patch["col"]), int(patch["patch_size"])
            lbl = lsrc.read(1, window=Window(c, r, ps, ps)).astype(np.uint8).ravel()
            X_p, _ = extract_patch_features(r, c, ps)
            if is_regressor:
                pred = np.clip(np.round(model.predict(X_p)), 0, 4).astype(np.uint8)
            else:
                pred = model.predict(X_p).astype(np.uint8)
            all_pred.append(pred)
            all_true.append(lbl)
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    acc_exact = float(np.mean(y_pred == y_true))
    acc_pm1   = accuracy_pm1(y_true, y_pred)
    print(f"\n[{model_name}] Exact accuracy: {acc_exact:.4f}  |  Accuracy±1: {acc_pm1:.4f}")
    return y_true, y_pred, acc_exact, acc_pm1

results = {}
for name, model in models.items():
    is_reg = (name == "hgbr")
    y_true, y_pred, acc, apm1 = run_val(model, name, is_regressor=is_reg)
    results[name] = {"y_true": y_true, "y_pred": y_pred, "acc": acc, "acc_pm1": apm1}

In [ ]:
# Confusion matrices
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6))
if n_models == 1:
    axes = [axes]
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(res["y_true"], res["y_pred"], labels=range(5))
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}\nExact={res['acc']:.3f}  Acc±1={res['acc_pm1']:.3f}")
plt.suptitle("Confusion matrices — validation set", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrices.png", dpi=120)
plt.show()
print(f"Saved → {OUT_DIR}/confusion_matrices.png")

# Summary table
print("\n=== VAL SUMMARY ===")
for name, res in results.items():
    print(f"  {name:10s}  exact={res['acc']:.4f}  acc±1={res['acc_pm1']:.4f}")
best = max(results, key=lambda k: results[k]["acc_pm1"])
print(f"\nBest model: {best}  (Acc±1={results[best]['acc_pm1']:.4f})")
if results[best]["acc_pm1"] >= 0.90:
    print("✓ Acc±1 ≥ 0.90 → this model can go directly to Notebook 5 (submission).")
else:
    print("✗ Acc±1 < 0.90 → proceed to Notebook 3 (U-Net) for improvement.")

## 6. Feature Importance

In [ ]:
# ExtraTrees gives clean impurity-based importance
if "etc" in models:
    imp = models["etc"].feature_importances_
    top_k = 25
    top_idx = np.argsort(imp)[-top_k:][::-1]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(top_k), imp[top_idx][::-1], color="steelblue")
    ax.set_yticks(range(top_k))
    ax.set_yticklabels([feat_names[i] for i in top_idx[::-1]], fontsize=8)
    ax.set_xlabel("Mean decrease in impurity")
    ax.set_title(f"Top {top_k} features — ExtraTrees")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "feature_importance.png", dpi=120)
    plt.show()
    print(f"Saved → {OUT_DIR}/feature_importance.png")
    print(f"\nTop 10 features: {[feat_names[i] for i in top_idx[:10]]}")

## 7. Patch-Level Prediction Demo on Val

In [ ]:
# Visual comparison: ground truth vs predicted mask for 5 val patches
from PIL import Image

best_model = models[best]
is_best_reg = (best == "hgbr")
cmap = mcolors.ListedColormap(CLASS_COLORS)
norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

VIS_N_PATCHES = 5
fig, axes = plt.subplots(VIS_N_PATCHES, 2, figsize=(8, 3 * VIS_N_PATCHES))

with rasterio.open(label_ref) as lsrc:
    for row_i, patch in enumerate(val_rows[:VIS_N_PATCHES]):
        r, c, ps = int(patch["row"]), int(patch["col"]), int(patch["patch_size"])
        lbl = lsrc.read(1, window=Window(c, r, ps, ps)).astype(np.uint8)
        X_p, _ = extract_patch_features(r, c, ps)
        if is_best_reg:
            pred = np.clip(np.round(best_model.predict(X_p)), 0, 4).astype(np.uint8)
        else:
            pred = best_model.predict(X_p).astype(np.uint8)
        pred_img = pred.reshape(ps, ps)
        apm1 = accuracy_pm1(lbl.ravel(), pred)

        axes[row_i, 0].imshow(lbl, cmap=cmap, norm=norm, interpolation="nearest")
        axes[row_i, 0].set_title(f"GT  patch {patch['patch_id']}", fontsize=8)
        axes[row_i, 0].axis("off")
        axes[row_i, 1].imshow(pred_img, cmap=cmap, norm=norm, interpolation="nearest")
        axes[row_i, 1].set_title(f"Pred  Acc±1={apm1:.3f}", fontsize=8)
        axes[row_i, 1].axis("off")

        # Save individual prediction PNG for inspection
        Image.fromarray(pred_img).save(VIS_DIR / f"{patch['patch_id']}_pred.png")

plt.suptitle(f"Val predictions ({best})", fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "val_patch_demo.png", dpi=120)
plt.show()
print(f"Saved → {OUT_DIR}/val_patch_demo.png")

## 8. Save Best Model Metadata for Notebook 5

**HITL handoff:** Review the confusion matrices and visual predictions above.

Key questions:
1. Is Acc±1 on val ≥ 0.90? If yes → run Notebook 5 directly.
2. Which classes have low recall? This guides class-weight tuning in Notebook 3.
3. Do the predicted masks look spatially smooth, or noisy? Noisy → Notebook 4's median filter.
4. Which features are most important? Share with Claude to refine seasonal groupings.

In [ ]:
meta_out = {
    "best_model": best,
    "acc_exact": results[best]["acc"],
    "acc_pm1": results[best]["acc_pm1"],
    "model_path": str(CKPT_DIR / f"{best}_pixel.pkl"),
    "features": feat_names,
    "n_features": len(feat_names),
    "pixels_per_class": PIXELS_PER_CLASS,
    "n_timeframes": n_times,
    "label_name": LABEL_NAME,
    "data_dir": DATA_DIR,
    "proceed_to_unet": results[best]["acc_pm1"] < 0.90,
}
(OUT_DIR / "baseline_meta.json").write_text(json.dumps(meta_out, indent=2))
print("Saved baseline_meta.json")
print(json.dumps({k: v for k, v in meta_out.items() if k != "features"}, indent=2))